# Day 12 — Solution: The Law of Large Numbers

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
import os
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

## E1 — convergence, thin vs fat

In [ ]:
rng = np.random.default_rng(11)
n = 10_000
thin = rng.normal(0.0005, 0.011, n)
fat = 0.0005 + 0.011 / np.sqrt(5 / 3) * rng.standard_t(3, n)  # scaled ~same σ

for name, series in [("normal", thin), ("t(3)", fat)]:
    m = np.cumsum(series) / np.arange(1, n + 1)
    se = series.std() / np.sqrt(np.arange(1, n + 1))
    plt.plot(m, label=name)
    plt.plot(m + 2 * se, ls=":", color="gray"); plt.plot(m - 2 * se, ls=":", color="gray")
plt.xscale("log"); plt.legend(); plt.title("Running means: thin vs fat tails")
plt.show()

**Expected reasoning.** Both converge — LLN needs only finite variance,
and t(3) has it (barely). But the t(3) path converges *violently*: long
plateaus punctuated by sudden jumps (one huge draw reshapes the running
mean even at n=5,000), while the normal path is a smooth glide. **With
fat tails, "wait for more data" buys you less certainty per day than
the SE formula advertises** — the effective sample size grows slower.
That's the honest version of "the mean is hard to estimate."

## E2 — days-to-resolve arithmetic

In [ ]:
for mu, sigma in [(0.0005, 0.011), (0.001, 0.011), (0.0005, 0.022)]:
    n = (2 * sigma / mu) ** 2
    print(f"mu={mu}, sigma={sigma}: n = {n:,.0f} days = {n/252:.1f} years")

(0.0005, 0.011): ~1,936 days ≈ 7.7 years. (0.001, 0.011): ~484 days ≈
1.9 years. (0.0005, 0.022): ~7,744 days ≈ 30.7 years. **Doubling the
edge quarters the time; doubling the vol quadruples it.** You can rarely
double an edge, but you can often halve a strategy's noise (netting,
hedging, better instruments) — engineering the denominator is usually
the cheaper path to statistical resolution. (Both are usually expensive:
that's the point.)

## E3 — the mean is barely resolved; vol is easy

In [ ]:
if DATA_SOURCE == "real":
    px = get_prices("SPY", start="1990-01-01")
else:
    px = synthetic_prices(n_days=8000, n_assets=1, seed=22, mu=0.0004)
    px.columns = ["SPY"]
r = px["SPY"].pct_change().dropna()
n = len(r); m, s = r.mean(), r.std()
t = m / (s / np.sqrt(n))
n_needed = (2 * s / m) ** 2
n_vol = 50   # for ±10% of sigma: 1/sqrt(2n) = 0.10 -> n = 50
print(f"n={n}, t-stat {t:.2f}, days needed for t=2: {n_needed:,.0f} ({n_needed/252:.0f}y)")
print(f"days for vol ±10%: ~{n_vol} (two months)")

Real SPY since 1990: t-stat of the daily mean ≈ 1.5–2 over ~8,800 days;
reaching t=2 needs on the order of 10,000+ days from scratch. Vol, by
contrast, is within ±10% after ~50 days. **Contrast sentence: markets
hand you volatility for free in two months and make you pay a decade for
the mean — so any claim about expected returns is automatically ~100×
more fragile than the same claim about risk.** (Exact means and years
vary with sample and data source; the order of magnitude is the result.)

## E4 — dilution, not compensation

In [ ]:
rng = np.random.default_rng(12)
flips = rng.random((10_000, 100)) < 0.5
bad_start = flips[:, :10].sum(axis=1) == 0
paths = flips[bad_start]
rem = paths[:, 10:].mean(axis=1)     # remaining average
overall = paths.mean(axis=1)         # overall average
print(f"paths starting 0-for-10: {bad_start.sum()}")
print(f"remaining avg (day 11+): mean {rem.mean():.3f}")
print(f"overall avg (incl. start): mean {overall.mean():.3f} "
      f"-> converges to 0.5 from below, by dilution")

After 0-for-10: the *remaining* flips average 50% (the coin has no
memory — no compensation ever arrives), while the *overall* average
climbs from 0% toward 50% purely because 90 neutral flips outvote the
bad start. **"Reversion" in the overall average is arithmetic dilution,
never a correcting force. Traders who expect the market to "owe them"
after a losing streak are confusing a statistic with a causal agent.**